# Tutorial: Single-Qubit Basics in pysuqu

Audience:
- Users who want a compact public example for the `pysuqu.qubit` surface.

Prerequisites:
- A Python environment with `numpy`, `matplotlib`, and `qutip` installed.
- Basic familiarity with transmon terminology and Jupyter notebooks.

Learning goals:
- Instantiate a public single-qubit model.
- Read spectrum information from the package-level analysis helpers.
- Sweep flux with the structured sweep result surface.


## Outline

1. Check the local environment and import the public qubit APIs.
2. Build a `GroundedTransmon` and inspect its spectrum.
3. Compare against a direct `AbstractQubit` model.
4. Sweep flux offsets and plot the resulting energy ladder.
5. Try a short exercise that changes the operating point.


In [ ]:
from __future__ import annotations

import importlib.util
from contextlib import redirect_stdout
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np

required = ["numpy", "matplotlib", "qutip"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(
        "Install the runtime dependencies first: " + ", ".join(missing)
    )

from pysuqu.qubit import AbstractQubit, GroundedTransmon
from pysuqu.qubit.analysis import analyze_single_qubit_spectrum
from pysuqu.qubit.sweeps import sweep_single_qubit_energy_vs_flux_base_result


## Step 1 - Build a grounded transmon

This public constructor is the fastest way to reach a concrete single-qubit model.
The snippet below suppresses constructor prints so the notebook stays compact.


In [ ]:
with redirect_stdout(StringIO()):
    transmon = GroundedTransmon(
        capacitance=80e-15,
        junction_resistance=10_000,
        inductance=1e20,
        flux=0.12,
        trunc_ener_level=4,
        junc_ratio=1.0,
        qr_couple=[3e-15],
    )

spectrum = analyze_single_qubit_spectrum(transmon)

{
    "f01_ghz": float(spectrum.f01),
    "anharmonicity_ghz": float(spectrum.anharmonicity),
    "flux_matrix": np.asarray(transmon.get_element_matrices("flux")).tolist(),
    "energy_matrix_keys": sorted(transmon.get_energy_matrices().keys()),
}


## Step 2 - Compare with `AbstractQubit`

`AbstractQubit` is useful when you already know the target frequency and anharmonicity.
It is a convenient reference model for scripting and lightweight checks.


In [ ]:
with redirect_stdout(StringIO()):
    reference = AbstractQubit(
        frequency=5e9,
        anharmonicity=-250e6,
        frequency_max=6e9,
        qubit_type="Transmon",
        energy_trunc_level=6,
        is_print=False,
    )

level_energies_ghz = [
    float(reference.get_energylevel(level) / (2 * np.pi))
    for level in range(1, 4)
]

{
    "reference_levels_ghz": level_energies_ghz,
    "f01_from_public_property_ghz": float(reference.qubit_f01),
    "anharmonicity_from_public_property_ghz": float(reference.qubit_anharm),
}


## Step 3 - Sweep flux offsets with the structured helper

The sweep helper returns a `SweepResult` object whose `series` field is ready for plotting
or downstream processing.


In [ ]:
flux_offsets = [np.array([[offset]]) for offset in np.linspace(-0.2, 0.3, 11)]
sweep = sweep_single_qubit_energy_vs_flux_base_result(
    transmon,
    flux_offsets,
    upper_level=3,
)

offset_values = [float(item[0, 0]) for item in sweep.sweep_values]

fig, ax = plt.subplots(figsize=(6, 4))
for label, values in sweep.series.items():
    ax.plot(offset_values, values, marker="o", label=label)

ax.set_xlabel("Flux offset (Phi0 units in this synthetic example)")
ax.set_ylabel("Transition energy (GHz)")
ax.set_title("Single-qubit energy response to flux offsets")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

{
    "series_keys": list(sweep.series.keys()),
    "restored_flux": np.asarray(transmon.get_element_matrices("flux")).tolist(),
}


## Exercises

- Change `flux=0.12` to `flux=0.18` and see how `f01` shifts.
- Increase `upper_level` to inspect more excited states.
- Common pitfall: mixing SI units and GHz-style display values in the same variable name.


In [ ]:
def summarize_f01_for_flux(flux_value: float) -> dict[str, float]:
    with redirect_stdout(StringIO()):
        model = GroundedTransmon(
            capacitance=80e-15,
            junction_resistance=10_000,
            inductance=1e20,
            flux=flux_value,
            trunc_ener_level=4,
            junc_ratio=1.1,
            qr_couple=[3e-15],
            is_print=False,
        )
    spectrum = analyze_single_qubit_spectrum(model)
    return {
        "flux": flux_value,
        "f01_ghz": float(spectrum.f01),
        "anharmonicity_ghz": float(spectrum.anharmonicity),
    }

[summarize_f01_for_flux(value) for value in (0.08, 0.12, 0.16)]
